# CAMRA — Notebook 3: Figures

Regenerates every figure of the paper from `results/`, with no training and no dataset needed. Set the paths in the configuration cell and run all cells; the figures are written to `figures/`.

In [ ]:
"""
Regenerate every manuscript figure from the saved results of the final base run (results/base).
No training. Set the three paths in the next cell, then run all cells.
"""
import sys, json, glob, os
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score


In [ ]:
# ---------------------------------------------------------------- paths (edit these)
RUN     = "../results/base"                     # folder containing reports/, artifacts/, manifests/
FP_JSON = "../results/base/reports/focal_prior_control.json"   # Focal Loss + Prior control results
OUT     = "../figures"                          # figures are written here

for _p in (RUN, FP_JSON):
    assert os.path.exists(_p), f"path not found: {_p}"
os.makedirs(OUT, exist_ok=True)
REP, ART, MAN = f"{RUN}/reports", f"{RUN}/artifacts", f"{RUN}/manifests"

# ---------------------------------------------------------------- style
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"], "mathtext.fontset": "dejavuserif",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "savefig.dpi": 300, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02, "axes.grid": False, "legend.frameon": False,
})
OK = dict(blue="#0072B2", verm="#D55E00", green="#009E73", purple="#CC79A7",
          orange="#E69F00", sky="#56B4E9", yellow="#F0E442", grey="#7F7F7F", black="#000000")
CLASSES = ["Benign", "DarkNexus", "Gafgyt", "Generic", "Mirai", "Other"]
CLS_COL = dict(Benign=OK["blue"], DarkNexus=OK["verm"], Gafgyt=OK["green"],
               Generic=OK["purple"], Mirai=OK["orange"], Other=OK["sky"])
CLS_LS = dict(Benign="-", DarkNexus="--", Gafgyt="-.", Generic=":", Mirai=(0, (5, 1)), Other=(0, (3, 1, 1, 1)))
W1, H1 = 3.3, 2.5          # one panel at ~0.48\linewidth
WFULL = 6.6                # full text width
SEED_TXT = "representative seed"


In [ ]:

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.show(); plt.close(fig); print("saved", name)

def pop_sd(x): return float(np.std(np.asarray(x, float)))

# ---------------------------------------------------------------- data
M = json.load(open(f"{REP}/multiseed_results.json"))
raw = M["raw_per_seed"]
PD = json.load(open(glob.glob(f"{ART}/plot_data*.json")[0]))
FPC = json.load(open(FP_JSON))
raw["Focal Loss + Prior"] = {k: [s[k] for s in FPC["per_seed"]] for k in ("macro_f1", "macro_f1_all", "weighted_f1", "accuracy")}
SHORT = {"Deep Reinforcement Learning": "DRL (CAMRA)", "Federated Learning": "FL",
         "Focal Loss + Prior": "Focal Loss + Prior$^{\\dagger}$"}
GROUP = {"Deep Reinforcement Learning": "w", "Federated Learning": "w", "Fixed Weighting": "w",
         "Focal Loss": "w", "Focal Loss + Prior": "w", "CNN": "b", "MLP": "b", "XGBoost": "b",
         "Random Forest": "b"}
GCOL = {"w": OK["blue"], "b": OK["grey"], "m": OK["green"]}
GLAB = {"w": "Full model / weighting schemes", "b": "Baselines", "m": "Modality subsets"}
rows = [k for k in raw if not k.startswith("_")]
mean = {k: np.mean(raw[k]["macro_f1"]) for k in rows}
order = sorted(rows, key=lambda k: -mean[k])


In [ ]:

# ================================================================ 1. ablation bar (horizontal)
fig, ax = plt.subplots(figsize=(WFULL, 4.2))
y = np.arange(len(order))[::-1]
for yi, k in zip(y, order):
    g = GROUP.get(k, "m"); m, s = mean[k], pop_sd(raw[k]["macro_f1"])
    ax.barh(yi, m, color=GCOL[g], edgecolor="black", linewidth=0.4, height=0.7,
            hatch="////" if k == "Focal Loss + Prior" else None)
    ax.errorbar(m, yi, xerr=s, color="black", capsize=2, lw=0.7, capthick=0.7)
    ax.text(m + s + 0.008, yi, f"{m:.3f}", va="center", fontsize=7)
ax.set_yticks(y); ax.set_yticklabels([SHORT.get(k, k) for k in order])
ax.set_xlabel("Macro-F1"); ax.set_xlim(0, 0.86); ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(facecolor=GCOL[g], edgecolor="black", lw=0.4, label=GLAB[g]) for g in "wbm"],
          loc="lower right", frameon=True, framealpha=1, edgecolor="0.8")
save(fig, "ablation_bar")


In [ ]:

# ================================================================ 2. macro vs weighted
order_mw = [k for k in order if k != "Focal Loss + Prior"]
fig, ax = plt.subplots(figsize=(WFULL, 2.9))
x = np.arange(len(order_mw)); w = 0.38
mf = [mean[k] for k in order_mw]; ms = [pop_sd(raw[k]["macro_f1"]) for k in order_mw]
wf = [np.mean(raw[k]["weighted_f1"]) for k in order_mw]; ws = [pop_sd(raw[k]["weighted_f1"]) for k in order_mw]
ax.bar(x - w/2, mf, w, yerr=ms, color=OK["blue"], edgecolor="black", lw=0.4, capsize=1.5, error_kw=dict(lw=0.6), label="Macro-F1")
ax.bar(x + w/2, wf, w, yerr=ws, color=OK["verm"], edgecolor="black", lw=0.4, capsize=1.5, error_kw=dict(lw=0.6), label="Weighted-F1")
lab = [SHORT.get(k, k).replace(" (CAMRA)", "") for k in order_mw]
ax.set_xticks(x); ax.set_xticklabels(lab, rotation=40, ha="right"); ax.tick_params(axis="x", length=0)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.0)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
save(fig, "macro_vs_weighted")


In [ ]:

# ================================================================ 3. central curves
H = PD["central_histories"]
def band(ax, key, col, ls, lab):
    a = np.array([h[key] for h in H], float); m, s = a.mean(0), a.std(0); e = np.arange(1, a.shape[1] + 1)
    ax.plot(e, m, color=col, ls=ls, label=lab); ax.fill_between(e, m - s, m + s, color=col, alpha=0.18, lw=0)
for key, ylab, name in (("acc", "Accuracy", "central_accuracy"), ("loss", "Loss", "central_loss")):
    fig, ax = plt.subplots(figsize=(W1, H1))
    band(ax, f"train_{key}", OK["blue"], "-", "Training")
    band(ax, f"val_{key}", OK["verm"], "--", "Validation (val\\_select)".replace("\\_", "_"))
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylab); ax.set_xlim(1, 50)
    ax.legend(loc="lower right" if key == "acc" else "upper right")
    save(fig, name)


In [ ]:

# ================================================================ 4. RL reward + multipliers
d = np.load(glob.glob(f"{ART}/rl_dynamics*.npz")[0], allow_pickle=True)
n = int(d["n_central_steps"]); R = np.asarray(d["reward_history"][:n], float)
e = np.arange(1, n + 1); ma = pd.Series(R).rolling(5, min_periods=1).mean().values
fig, ax = plt.subplots(figsize=(W1, H1))
ax.plot(e, R, color=OK["sky"], lw=0.9, marker="o", ms=2, label="Per-epoch reward $R_t$")
ax.plot(e, ma, color=OK["verm"], lw=1.4, label="5-epoch moving average")
ax.axhline(0, color="0.5", lw=0.6, ls=":")
ax.set_xlabel("Epoch"); ax.set_ylabel("Reward"); ax.set_xlim(1, n); ax.legend(loc="lower right")
save(fig, "rl_rewards_central")

names = [str(c) for c in d["class_names"]]
man = json.load(open(glob.glob(f"{MAN}/preprocessing_manifest*.json")[0]))
prior = np.array([man["class_weights"][c] for c in names])
Mx = np.asarray(d["weight_history"][:n], float) / prior
pz = np.load(glob.glob(f"{ART}/policy_diagnostics*.npz")[0], allow_pickle=True)
ncp = int(pz["n_central_steps"]) if "n_central_steps" in pz.files else n
A = np.asarray(pz["alpha"][:ncp], float); pol_mean = (len(names) * A / A.sum(1, keepdims=True)).mean(0)
show = [c for c in names if c != "Other"]
fig, ax = plt.subplots(figsize=(W1, H1))
rng_j = np.random.default_rng(1)
for i, c in enumerate(show):
    col = Mx[:, names.index(c)]
    ax.boxplot(col, positions=[i], widths=0.5, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="white", edgecolor="0.3", lw=0.7), medianprops=dict(color="0.3", lw=0.9),
               whiskerprops=dict(color="0.3", lw=0.7), capprops=dict(color="0.3", lw=0.7))
    ax.scatter(i + rng_j.uniform(-0.17, 0.17, len(col)), col, s=7, color=CLS_COL[c], alpha=0.8, lw=0, zorder=3)
    ax.scatter(i, pol_mean[names.index(c)], marker="D", s=22, color="black", zorder=4)
ax.axhline(1, color="black", lw=0.6, ls=":", zorder=0)
ax.set_yscale("log"); ax.set_yticks([0.4, 0.5, 0.7, 1, 1.5, 2]); ax.set_yticklabels(["0.4", "0.5", "0.7", "1", "1.5", "2"])
ax.minorticks_off(); ax.set_xticks(range(len(show))); ax.set_xticklabels(show, rotation=20, ha="right")
ax.tick_params(axis="x", length=0, top=False); ax.set_ylabel("Multiplier $m_{c,t}$ (log scale)")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color="0.4", ms=3, label="Sampled (one per epoch)"),
                   Line2D([], [], marker="D", ls="", color="black", ms=4, label="Policy mean")],
          loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, columnspacing=1.0)
save(fig, "rl_weights_central")


In [ ]:

# ================================================================ 5. policy diagnostics (3 panels)
p = np.load(glob.glob(f"{ART}/policy_diagnostics*.npz")[0], allow_pickle=True)
nc = int(p["n_central_steps"]) if "n_central_steps" in p.files else n
a0, ent, gap = (np.asarray(p[k][:nc], float) for k in ("alpha0", "entropy", "explore_gap"))
st = np.arange(1, nc + 1); K = len(names)
PW = (2.25, 1.9)
fig, ax = plt.subplots(figsize=PW)
ax.plot(st, a0, color=OK["blue"]); ax.axhline(60, color="0.4", ls="--", lw=0.8, label="Lower bound (60)")
ax.set_ylim(59, max(a0.max(), 60) + 1); ax.set_xlabel("Policy update (epoch)"); ax.set_ylabel(r"$\sum_c \alpha_c$")
ax.set_xlim(1, nc); ax.legend(loc="center right"); save(fig, "policy_alpha0")
fig, ax = plt.subplots(figsize=PW)
ax.plot(st, ent, color=OK["verm"]); ax.set_xlabel("Policy update (epoch)"); ax.set_ylabel("Dirichlet entropy")
ax.set_xlim(1, nc); save(fig, "policy_entropy")
rng = np.random.default_rng(0)
exp = np.array([np.abs(rng.dirichlet(np.full(K, a / K), 20000) - 1 / K).sum(1).mean() for a in a0])
fig, ax = plt.subplots(figsize=PW)
ax.plot(st, gap, color=OK["green"], label="Sampled"); ax.plot(st, exp, color="0.4", ls="--", lw=0.8, label="Uniform policy")
ax.set_xlabel("Policy update (epoch)"); ax.set_ylabel("Sampling spread ($\\ell_1$)"); ax.set_xlim(1, nc)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, columnspacing=0.8, handlelength=1.5)
save(fig, "policy_explore_gap")


In [ ]:

# ================================================================ 6. FL client drift
CF = PD["client_f1_history"]; ARCH = ["ARM", "MIPS", "MIPSEL", "x86"]
ACOL = dict(ARM=OK["blue"], MIPS=OK["verm"], MIPSEL=OK["green"], x86=OK["purple"])
ALS = dict(ARM="-", MIPS="--", MIPSEL="-.", x86=":")
roll = {a: pd.Series(np.asarray(CF[a], float)).rolling(5, center=True, min_periods=1).mean().values for a in ARCH}
r = np.arange(1, len(CF["ARM"]) + 1)
fig, ax = plt.subplots(figsize=(W1, H1))
for a in ARCH: ax.plot(r, roll[a], color=ACOL[a], ls=ALS[a], label=a)
ax.set_xlabel("Federated round"); ax.set_ylabel("Macro-F1 (5-round rolling mean)"); ax.set_xlim(1, r[-1])
ax.legend(loc="lower right", ncol=2)
save(fig, "fl_drift_all_archs_rolling")
allv = np.concatenate([np.asarray(CF[a], float) for a in ARCH]); lo, hi = allv.min() - 0.01, allv.max() + 0.01
for a in ARCH:
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.plot(r, CF[a], color="0.55", lw=0.8, marker="o", ms=1.8, label="Per round")
    ax.plot(r, roll[a], color=ACOL[a], lw=1.4, label="5-round rolling mean")
    ax.set_ylim(lo, hi); ax.set_xlim(1, r[-1]); ax.set_xlabel("Federated round"); ax.set_ylabel("Macro-F1")
    ax.legend(loc="lower right")
    save(fig, f"fl_drift_{a.lower()}")


In [ ]:

# ================================================================ 7. FL global curves
F = PD["fl_histories"]
for key, ylab, name, col in (("global_acc", "Accuracy", "fl_global_accuracy", OK["blue"]),
                             ("global_loss", "Loss", "fl_global_loss", OK["verm"])):
    a = np.array([h[key] for h in F], float); m, s = a.mean(0), a.std(0); rr = np.arange(1, a.shape[1] + 1)
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.plot(rr, m, color=col); ax.fill_between(rr, m - s, m + s, color=col, alpha=0.18, lw=0)
    ax.set_xlabel("Federated round"); ax.set_ylabel(ylab); ax.set_xlim(1, rr[-1])
    save(fig, name)


In [ ]:

# ================================================================ 8. confusion matrices (row-normalised colour, counts)
cmap = LinearSegmentedColormap.from_list("okb", ["#FFFFFF", "#9ECAE1", OK["blue"]])
for src, name in (("central", "cm_central"), ("fl_global", "cm_fl_global")):
    z = np.load(f"{ART}/{src}.npz", allow_pickle=True)
    cm = confusion_matrix(z["y_true"], z["y_pred"], labels=range(len(CLASSES)))
    rown = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(W1, W1 * 0.92))
    im = ax.imshow(rown, cmap=cmap, vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=7,
                    color="white" if rown[i, j] > 0.6 else "black")
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=40, ha="right"); ax.set_yticklabels(CLASSES)
    ax.tick_params(length=0, top=False, right=False); ax.minorticks_off()
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.set_label("Fraction of true class"); cb.ax.tick_params(labelsize=6)
    save(fig, name)


In [ ]:

# ================================================================ 9. ROC / PR (Other omitted: one test sample)
CURVE_CLS = [c for c in CLASSES if c != "Other"]
for src, tag in (("central", "central"), ("fl_global", "fl_global")):
    z = np.load(f"{ART}/{src}.npz", allow_pickle=True); yt, pr = z["y_true"], z["y_prob"]
    for kind in ("roc", "pr"):
        fig, ax = plt.subplots(figsize=(W1, H1 + 0.35))
        for c in CURVE_CLS:
            i = CLASSES.index(c); yb = (yt == i).astype(int)
            if kind == "roc":
                fpr, tpr, _ = roc_curve(yb, pr[:, i]); v = auc(fpr, tpr)
                ax.plot(fpr, tpr, color=CLS_COL[c], ls=CLS_LS[c], label=f"{c} ({v:.2f})")
            else:
                pp, rc, _ = precision_recall_curve(yb, pr[:, i]); v = average_precision_score(yb, pr[:, i])
                ax.step(rc, pp, where="post", color=CLS_COL[c], ls=CLS_LS[c], label=f"{c} ({v:.2f})")
        if kind == "roc":
            ax.plot([0, 1], [0, 1], color="0.6", lw=0.6, ls=":"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ttl = "Class (AUC)"
        else:
            ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ttl = "Class (AP)"
        ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.02)
        ax.legend(title=ttl, title_fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3,
                  columnspacing=0.9, handlelength=2.0)
        save(fig, f"{tag}_{kind}")


In [ ]:

# ================================================================ 10. modality holdout
comp = np.asarray(raw["Deep Reinforcement Learning"]["macro_f1"], float)
mh = M["modality_holdout"]
labs = ["Complete", "Without PCAP", "Without SAR", "Without STRACE"]
vals = [comp.mean()] + [mh[k]["macro_f1_mean"] for k in ("PCAP", "SAR", "STRACE")]
sds = [comp.std()] + [mh[k]["macro_f1_std"] * np.sqrt(9 / 10) for k in ("PCAP", "SAR", "STRACE")]  # stored with n-1 -> population
fig, ax = plt.subplots(figsize=(W1, H1))
cols = [OK["blue"]] + [OK["verm"]] * 3
ax.bar(range(4), vals, yerr=sds, color=cols, edgecolor="black", lw=0.4, capsize=3, error_kw=dict(lw=0.7), width=0.65)
for i, (v, s) in enumerate(zip(vals, sds)): ax.text(i, v + s + 0.02, f"{v:.3f}", ha="center", fontsize=7)
ax.set_xticks(range(4)); ax.set_xticklabels(labs, rotation=15, ha="right"); ax.tick_params(axis="x", length=0)
ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 0.9)
save(fig, "modality_holdout")


In [ ]:

# ================================================================ 11. UMAP (no in-plot boxes; shared legend strip)
u = np.load(f"{ART}/umap_panels.npz", allow_pickle=True)
draw_order = ["Benign", "Mirai", "DarkNexus", "Gafgyt", "Generic", "Other"]   # rare classes drawn last
SIZE = dict(Benign=3, Mirai=3, DarkNexus=9, Gafgyt=9, Generic=9, Other=14)
for key in ("umap_raw_pcap", "umap_raw_sar", "umap_raw_strace", "umap_embeddings"):
    emb, lab = u[f"{key}__emb"], u[f"{key}__lab"]
    fig, ax = plt.subplots(figsize=(W1, W1 * 0.8))
    for c in draw_order:
        mk = lab == CLASSES.index(c)
        if mk.any():
            ax.scatter(emb[mk, 0], emb[mk, 1], s=SIZE[c], color=CLS_COL[c], alpha=0.75 if SIZE[c] < 5 else 0.95,
                       linewidths=0.3 if SIZE[c] >= 5 else 0, edgecolors="black" if SIZE[c] >= 5 else "none")
    ax.set_xticks([]); ax.set_yticks([]); ax.tick_params(top=False, right=False)
    save(fig, key)
fig = plt.figure(figsize=(WFULL, 0.3))
fig.legend(handles=[Line2D([], [], marker="o", ls="", color=CLS_COL[c], markeredgecolor="black" if SIZE[c] >= 5 else CLS_COL[c],
                           markeredgewidth=0.3, ms=5, label=c) for c in CLASSES],
           loc="center", ncol=6, frameon=False)
save(fig, "umap_legend")
print("done")

In [ ]:

# ================================================================ 12. Dataset: executions per architecture x class (Fig. 3)
# Built from the split manifest (one row per aligned execution = one unique file hash per architecture),
# i.e. the curated dataset after the rare families were merged into 'Other'.
mf = pd.read_csv(glob.glob(f"{MAN}/split_manifest*.csv")[0])
ARCH_ORDER = ["ARM", "MIPS", "MIPSEL", "x86"]
ct = pd.crosstab(mf["arch"], mf["family"]).reindex(index=ARCH_ORDER, columns=CLASSES, fill_value=0)
print(ct, "\ntotal executions:", int(ct.values.sum()))

hm_cmap = LinearSegmentedColormap.from_list("okb_seq", ["#EAF3FB", "#9ECAE1", OK["blue"]])
hm_cmap.set_bad("white")                                   # zero cells (masked) stay white
vals = ct.values.astype(float)
fig, ax = plt.subplots(figsize=(4.8, 2.1))
im = ax.imshow(np.ma.masked_equal(vals, 0), cmap=hm_cmap, norm=LogNorm(vmin=1, vmax=vals.max()), aspect="auto")
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = int(vals[i, j])
        ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=7,
                color="white" if v > 300 else "black")
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels([f"{c}\n({int(ct[c].sum()):,})" for c in CLASSES])
ax.set_yticks(range(len(ARCH_ORDER)))
ax.set_yticklabels([f"{a} ({int(ct.loc[a].sum()):,})" for a in ARCH_ORDER])
ax.tick_params(length=0, top=False, right=False); ax.minorticks_off()
for s_ in ax.spines.values(): s_.set_linewidth(0.6)
ax.set_xticks(np.arange(-0.5, len(CLASSES), 1), minor=True); ax.set_yticks(np.arange(-0.5, len(ARCH_ORDER), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.2); ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("Executions (log scale)"); cb.ax.tick_params(labelsize=6)
save(fig, "eda_arch_class_heatmap")


In [ ]:
# ═══════════════════════════════════════════════════════════
# 13. FIG. 2 — Per-architecture sunburst from the log10 row counts of the raw captures
# (values computed from the raw parquet files; see STEP 4 of 01_dataset_creation.ipynb for the exact counts)
# Inner ring: data stream; outer ring: label within that stream (segment size = log value).
# ═══════════════════════════════════════════════════════════
import io, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

os.makedirs(OUT, exist_ok=True)   # same output folder as the other figures

LOG_VALUES = {
"x86": """PCAP,Benign,4.92
PCAP,DarkNexus,4
PCAP,Gafgyt,4
PCAP,Generic,4
PCAP,Mirai,6
PCAP,Unknown,5
SAR,Benign,5.49
SAR,DarkNexus,4
SAR,Gafgyt,2.96
SAR,Generic,3.16
SAR,Mirai,5.28
SAR,Unknown,4.31
STRACE,Benign,6.77
STRACE,DarkNexus,5.36
STRACE,Gafgyt,4.52
STRACE,Generic,4.2
STRACE,Mirai,7.3
STRACE,Unknown,0""",
"arm": """PCAP,Agent,1.41
PCAP,Benign,4.71
PCAP,DarkNexus,4.64
PCAP,Gafgyt,4.02
PCAP,Generic,3.96
PCAP,Mirai,5.71
PCAP,Tsunami,2.9
PCAP,Unknown,5.06
SAR,Agent,2.08
SAR,Benign,5.36
SAR,DarkNexus,4.02
SAR,Gafgyt,2.77
SAR,Generic,3.54
SAR,Mirai,5.51
SAR,Tsunami,2.56
SAR,Unknown,4.9
STRACE,Agent,1.3
STRACE,Benign,5.94
STRACE,DarkNexus,5.7
STRACE,Gafgyt,4.93
STRACE,Generic,5.17
STRACE,Mirai,7.31
STRACE,Tsunami,4.69
STRACE,Unknown,0""",
"mips": """PCAP,Benign,4.92
PCAP,DarkNexus,4.13
PCAP,Gafgyt,4.29
PCAP,Generic,3.1
PCAP,Mirai,5.78
PCAP,Unknown,5.17
SAR,Benign,5.41
SAR,DarkNexus,3.56
SAR,Gafgyt,3.26
SAR,Generic,2.25
SAR,Mirai,5.13
SAR,Unknown,4.53
STRACE,Benign,6.82
STRACE,DarkNexus,5.46
STRACE,Gafgyt,5.21
STRACE,Generic,3.7
STRACE,Mirai,7.36
STRACE,Unknown,0""",
"mipsel": """PCAP,Agent,2.53
PCAP,Benign,4.91
PCAP,DarkNexus,4.02
PCAP,Gafgyt,4.2
PCAP,Generic,3.2
PCAP,Mirai,5.76
PCAP,Rudedevil,2.32
PCAP,Unknown,5.63
SAR,Agent,1.98
SAR,Benign,5.4
SAR,DarkNexus,3.53
SAR,Gafgyt,3.2
SAR,Generic,2.53
SAR,Mirai,5.11
SAR,Rudedevil,1.93
SAR,Unknown,5.11
STRACE,Agent,3.8
STRACE,Benign,6.73
STRACE,DarkNexus,5.36
STRACE,Gafgyt,5.09
STRACE,Generic,4.08
STRACE,Mirai,7.34
STRACE,Rudedevil,2.45
STRACE,Unknown,0""",
}
df = pd.concat([pd.read_csv(io.StringIO(v), names=["stream", "label", "log"]).assign(arch=k)
                for k, v in LOG_VALUES.items()], ignore_index=True)

# ---------------------------------------------------------------- style (matches the other figures)
plt.rcParams.update({"font.family": "serif", "font.serif": ["DejaVu Serif"], "font.size": 8,
                     "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02})
STREAM_COLOUR = {"STRACE": "#009E73", "PCAP": "#D55E00", "SAR": "#E69F00"}   # Okabe-Ito
STREAM_TEXT   = {"STRACE": "white",   "PCAP": "white",   "SAR": "black"}
STREAM_ORDER  = ["STRACE", "PCAP", "SAR"]
LABEL_ORDER   = ["Mirai", "Benign", "Unknown", "DarkNexus", "Gafgyt", "Generic", "Tsunami", "Agent", "Rudedevil"]

def tint(c, a):
    r, g, b = to_rgb(c); return (r + (1 - r) * a, g + (1 - g) * a, b + (1 - b) * a)

def ring_text(ax, theta, r, text, colour, size, weight="normal", tangential=False):
    base = theta - 90 if tangential else theta
    rot = base if -90 <= (base + 180) % 360 - 180 <= 90 else base + 180
    t = np.deg2rad(theta)
    ax.text(r * np.cos(t), r * np.sin(t), text, rotation=rot, rotation_mode="anchor",
            ha="center", va="center", fontsize=size, color=colour, fontweight=weight)

def sunburst(arch):
    sub = df[(df.arch == arch) & (df.log > 0)]          # zero-size segments (e.g. STRACE Unknown) are omitted
    inner, outer = [], []
    for s in STREAM_ORDER:
        rows = sub[sub.stream == s].set_index("label")["log"]
        labs = [l for l in LABEL_ORDER if l in rows.index]
        inner.append((s, rows[labs].sum()))
        outer += [(s, l, rows[l]) for l in labs]

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    edge, start = dict(edgecolor="white", linewidth=1.0), 90
    ax.pie([v for _, v in inner], radius=0.60, colors=[STREAM_COLOUR[s] for s, _ in inner],
           startangle=start, counterclock=False, wedgeprops=dict(width=0.28, **edge))
    ax.pie([v for *_, v in outer], radius=1.0, colors=[tint(STREAM_COLOUR[s], 0.12) for s, *_ in outer],
           startangle=start, counterclock=False, wedgeprops=dict(width=0.40, **edge))

    tot, acc = sum(v for _, v in inner), 0.0
    for s, v in inner:
        ring_text(ax, start - 360 * (acc + v / 2) / tot, 0.46, s, STREAM_TEXT[s], 6.5, "bold", tangential=True); acc += v
    tot, acc = sum(v for *_, v in outer), 0.0
    for s, l, v in outer:
        ring_text(ax, start - 360 * (acc + v / 2) / tot, 0.80, l, STREAM_TEXT[s],
                  4.8 if 360 * v / tot > 9 else 4.2); acc += v
    ax.set(aspect="equal")
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUT}/{arch}.{ext}")
    plt.show(); plt.close(fig)

for arch in ["arm", "mips", "mipsel", "x86"]:
    sunburst(arch)                                       # plots/arm.pdf, mips.pdf, mipsel.pdf, x86.pdf
